# Applying dHT to Video Models (ViViT)

This notebook explores how to extend dHT tokenization to video understanding models like ViViT (Video Vision Transformer).

## Video Understanding Challenges

Video models face unique challenges:
- **Temporal dimension**: Need to process sequences of frames
- **Computational cost**: More data than single images
- **Redundancy**: Adjacent frames are often similar
- **Motion**: Need to capture temporal dynamics

## How dHT Helps

dHT can benefit video models by:
1. **Adaptive tokenization per frame**: Fewer tokens for static regions
2. **Temporal consistency**: Similar regions across frames can be merged
3. **Efficiency**: Reduce redundancy in static background
4. **Focus on motion**: More tokens for changing regions

## Approaches

1. **Frame-by-frame dHT**: Apply dHT independently to each frame
2. **Temporal dHT**: Extend tokenization to 3D (space + time)
3. **Hybrid**: dHT for spatial, fixed for temporal

In [ ]:
import torch
import torch.nn as nn
from dht.nn.transformer import dHTEncoder
from dht.tok.tokenizer import dHTTokenizer
from dht.tok.extractor import dHTExtractor
from dht.tok.embedder import dHTEmbedder
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Frame-by-Frame dHT

The simplest approach: apply dHT to each frame independently.

In [ ]:
class dHTVideoFramewise(nn.Module):
    def __init__(self, embed_dim=384, patch_size=16, num_frames=16):
        super().__init__()
        self.embed_dim = embed_dim
        self.patch_size = patch_size
        self.num_frames = num_frames
        
        # Spatial tokenization (per frame)
        self.tokenizer = dHTTokenizer(3, 8, compute_grad=True)
        self.extractor = dHTExtractor(patch_size)
        self.embedder = dHTEmbedder(embed_dim, patch_size, compute_grad=True)
        
        # Temporal embedding (for frame position)
        self.temporal_embed = nn.Parameter(torch.randn(1, num_frames, embed_dim) * 0.02)
        
        # Transformer for spatiotemporal processing
        from dht.nn.transformer import TransformerBlock
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, heads=6)
            for _ in range(12)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        
    def forward(self, video):
        """
        Args:
            video: [B, T, C, H, W] where T is num_frames
        """
        B, T, C, H, W = video.shape
        
        # Reshape to process all frames at once
        frames = video.view(B * T, C, H, W)
        
        # Apply dHT to all frames
        tok_result = self.tokenizer(frames)
        ext_result = self.extractor(tok_result)
        emb_result = self.embedder(ext_result)
        
        # Get tokens: [B*T, N, D] where N varies per frame
        tokens = emb_result.fV
        
        # Add temporal embeddings
        # Note: This is simplified; real implementation needs careful handling
        # of variable token counts per frame
        
        return tokens, emb_result

# Create model
video_model = dHTVideoFramewise().to(device)
print('Frame-wise dHT video model created')

In [ ]:
# Test with synthetic video
B, T, C, H, W = 2, 8, 3, 224, 224
test_video = torch.randn(B, T, C, H, W).to(device)

with torch.no_grad():
    video_model.eval()
    output, emb_result = video_model(test_video)

print(f'Input video: {test_video.shape}')
print(f'Output tokens: {output.shape}')
print(f'Tokens per frame (avg): {output.shape[1] // T}')

## 2. Analyzing Temporal Consistency

Visualize how tokenization changes across frames:

In [ ]:
def visualize_video_tokenization(video, tokenizer):
    """
    Visualize tokenization of video frames.
    """
    T = video.shape[1]
    
    fig, axes = plt.subplots(2, min(T, 4), figsize=(16, 8))
    
    for t in range(min(T, 4)):
        frame = video[0:1, t]  # Get single frame
        
        with torch.no_grad():
            result = tokenizer(frame)
        
        # Original frame
        axes[0, t].imshow(frame[0].permute(1, 2, 0).cpu().numpy())
        axes[0, t].set_title(f'Frame {t}')
        axes[0, t].axis('off')
        
        # Segmentation
        seg = result.seg[0].cpu().numpy()
        axes[1, t].imshow(seg, cmap='tab20')
        axes[1, t].set_title(f'{result.nV} tokens')
        axes[1, t].axis('off')
    
    plt.tight_layout()
    plt.show()

# Create video with motion
def create_moving_object_video(num_frames=8, size=224):
    video = torch.zeros(1, num_frames, 3, size, size)
    
    for t in range(num_frames):
        # Static background
        video[:, t, :, :, :] = 0.3
        
        # Moving object
        pos = int(t * size / num_frames)
        video[:, t, 0, pos:pos+30, 50:80] = 1.0  # Red rectangle
        video[:, t, 1, pos:pos+30, 50:80] = 0.0
        video[:, t, 2, pos:pos+30, 50:80] = 0.0
    
    return video

motion_video = create_moving_object_video().to(device)
visualize_video_tokenization(motion_video, video_model.tokenizer)

## 3. Temporal Token Merging

Merge tokens across time for static regions:

In [ ]:
class dHTVideoWithTemporalMerging(nn.Module):
    """
    Video model that merges tokens across time for static regions.
    """
    def __init__(self, embed_dim=384, patch_size=16, num_frames=16):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_frames = num_frames
        
        # Spatial tokenization
        self.spatial_tokenizer = dHTTokenizer(3, 8, compute_grad=True)
        self.extractor = dHTExtractor(patch_size)
        self.embedder = dHTEmbedder(embed_dim, patch_size, compute_grad=True)
        
        # Temporal processing
        self.temporal_attn = nn.MultiheadAttention(embed_dim, num_heads=8, batch_first=True)
        
    def merge_temporal_tokens(self, frame_tokens, threshold=0.9):
        """
        Merge similar tokens across consecutive frames.
        
        Args:
            frame_tokens: List of token tensors for each frame
            threshold: Similarity threshold for merging
        """
        # Simplified version - real implementation would:
        # 1. Compute token similarities across frames
        # 2. Merge tokens with high similarity
        # 3. Keep separate tokens for motion regions
        
        # For now, just concatenate
        return torch.cat(frame_tokens, dim=1)
    
    def forward(self, video):
        B, T, C, H, W = video.shape
        
        frame_tokens = []
        for t in range(T):
            frame = video[:, t]
            
            tok_result = self.spatial_tokenizer(frame)
            ext_result = self.extractor(tok_result)
            emb_result = self.embedder(ext_result)
            
            frame_tokens.append(emb_result.fV)
        
        # Merge tokens across time
        merged_tokens = self.merge_temporal_tokens(frame_tokens)
        
        return merged_tokens

temporal_model = dHTVideoWithTemporalMerging().to(device)
print('dHT video model with temporal merging created')

## 4. Efficiency Analysis

Compare token counts for static vs dynamic videos:

In [ ]:
# Create static video
static_video = torch.randn(1, 1, 3, 224, 224).expand(1, 8, 3, 224, 224).to(device)

# Create dynamic video
dynamic_video = torch.randn(1, 8, 3, 224, 224).to(device)

def count_tokens(video, model):
    with torch.no_grad():
        model.eval()
        tokens = model(video)
    return tokens.shape[1]

static_tokens = count_tokens(static_video, temporal_model)
dynamic_tokens = count_tokens(dynamic_video, temporal_model)

print('Token count comparison:')
print(f'  Static video: {static_tokens} tokens')
print(f'  Dynamic video: {dynamic_tokens} tokens')
print(f'  Reduction for static: {(1 - static_tokens/dynamic_tokens)*100:.1f}%')
print('\nFixed ViViT would use: ~1568 tokens (196 per frame × 8 frames)')

## 5. ViViT Architecture Variants

ViViT has several variants. Here's how dHT can be integrated:

In [ ]:
# Variant 1: Factorized Encoder (Spatial then Temporal)
class dHTViViTFactorized(nn.Module):
    def __init__(self, embed_dim=384, num_frames=8):
        super().__init__()
        
        # Spatial encoder with dHT
        self.spatial_encoder = dHTEncoder.build('S', patch_size=16)
        
        # Temporal transformer
        from dht.nn.transformer import TransformerBlock
        self.temporal_blocks = nn.ModuleList([
            TransformerBlock(embed_dim, heads=6) for _ in range(4)
        ])
        
    def forward(self, video):
        B, T, C, H, W = video.shape
        
        # Process each frame spatially
        spatial_features = []
        for t in range(T):
            frame = video[:, t]
            feat = self.spatial_encoder(frame)
            spatial_features.append(feat.fV[:, 0:1])  # CLS token
        
        # Stack temporal
        temporal_tokens = torch.cat(spatial_features, dim=1)  # [B, T, D]
        
        # Process temporally
        for block in self.temporal_blocks:
            temporal_tokens, _, _ = block(temporal_tokens, None, None)
        
        return temporal_tokens

print('Factorized ViViT with dHT created')

# Variant 2: Joint Space-Time (Single stream)
class dHTViViTJoint(nn.Module):
    def __init__(self, embed_dim=384):
        super().__init__()
        
        # Apply dHT to each frame
        self.frame_encoder = dHTVideoFramewise(embed_dim)
        
    def forward(self, video):
        # Process all frames jointly
        return self.frame_encoder(video)

print('Joint space-time ViViT with dHT created')

## 6. Practical Considerations

When using dHT with video models:

In [ ]:
# Memory management: Process frames in chunks
def process_long_video(video, model, chunk_size=4):
    """
    Process long videos in chunks to manage memory.
    """
    B, T, C, H, W = video.shape
    outputs = []
    
    for start in range(0, T, chunk_size):
        end = min(start + chunk_size, T)
        chunk = video[:, start:end]
        
        with torch.no_grad():
            output = model(chunk)
        outputs.append(output)
    
    return torch.cat(outputs, dim=1)

# Test chunked processing
long_video = torch.randn(1, 16, 3, 224, 224).to(device)
temporal_model.eval()
result = process_long_video(long_video, temporal_model, chunk_size=4)
print(f'Processed long video: {long_video.shape} -> {result.shape}')

## 7. Training Tips

Tips for training video models with dHT:

In [ ]:
print('Training Tips for dHT Video Models:')
print('\n1. Data Augmentation:')
print('   - Temporal jittering: Sample frames at different rates')
print('   - Spatial augmentation: Standard image augmentations')
print('   - Consistent augmentation across frames in same clip')
print('\n2. Optimization:')
print('   - Gradient checkpointing for memory efficiency')
print('   - Mixed precision training (fp16)')
print('   - Accumulate gradients if batch size is limited')
print('\n3. Training Strategy:')
print('   - Pre-train on images first (ImageNet)')
print('   - Fine-tune on video with frozen spatial encoder')
print('   - Gradually unfreeze for end-to-end training')
print('\n4. Hyperparameters:')
print('   - Lower learning rate for video (1e-5 to 1e-4)')
print('   - Longer warmup (5-10 epochs)')
print('   - Target token count: ~50-100 per frame')

## Summary

This notebook covered:
1. Frame-by-frame dHT application
2. Temporal consistency analysis
3. Temporal token merging strategies
4. Efficiency gains for video
5. ViViT architecture variants with dHT
6. Memory management for long videos
7. Training tips

### Key Benefits for Video:
- **Adaptive tokens per frame**: Efficient for static regions
- **Temporal merging**: Reduce redundancy
- **Motion focus**: More tokens where change occurs
- **Scalability**: Handle longer videos

### Challenges:
- Variable token counts across frames
- Temporal alignment of tokens
- Memory management
- Training complexity

### Future Directions:
- 3D tokenization (space + time jointly)
- Optical flow integration
- Multi-scale temporal modeling
- Efficient long-video understanding